<a href="https://colab.research.google.com/github/Jin-Miwacle/Metacognitive-Debugger/blob/T5_fine-tune/valhallafinetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Valhalla Fine-tune**

In [16]:
import json

examples = [
    {
        "passage_id": "garden_01",
        "language": "en",
        "context": "Daniel visited the community garden every Saturday. He helped move the plants near the fence where they received more sunlight.",
        "answer": "more sunlight",
        "question": "What did the plants receive near the fence?"
    },
    {
        "passage_id": "garden_01",
        "language": "en",
        "context": "Daniel visited the community garden every Saturday. He helped move the plants near the fence where they received more sunlight.",
        "answer": "every Saturday",
        "question": "When did Daniel visit the community garden?"
    }
]

print("Number of examples:", len(examples))
print(json.dumps(examples, indent=2, ensure_ascii=False))

Number of examples: 2
[
  {
    "passage_id": "garden_01",
    "language": "en",
    "context": "Daniel visited the community garden every Saturday. He helped move the plants near the fence where they received more sunlight.",
    "answer": "more sunlight",
    "question": "What did the plants receive near the fence?"
  },
  {
    "passage_id": "garden_01",
    "language": "en",
    "context": "Daniel visited the community garden every Saturday. He helped move the plants near the fence where they received more sunlight.",
    "answer": "every Saturday",
    "question": "When did Daniel visit the community garden?"
  }
]


In [15]:
def validate_examples(examples):
    errors = []

    for i, ex in enumerate(examples):
        required = ["passage_id", "language", "context", "answer", "question"]

        for field in required:
            if not ex.get(field, "").strip():
                errors.append(f"Example {i + 1}: Missing {field}")

        if ex.get("language") not in ["en", "fil"]:
            errors.append(f"Example {i + 1}: Language must be 'en' or 'fil'")

        context = ex.get("context", "")
        answer = ex.get("answer", "")
        question = ex.get("question", "")

        if answer and context.count(answer) != 1:
            errors.append(
                f"Example {i + 1}: Answer must appear exactly once in context"
            )

        if question and not question.endswith("?"):
            errors.append(f"Example {i + 1}: Question must end with '?'")

    if errors:
        print("Validation errors:")
        for error in errors:
            print("-", error)
    else:
        print("All examples passed basic validation!")

validate_examples(examples)

All examples passed basic validation!


In [13]:
# Combine both datasets
all_examples = examples + english_examples

print("Total examples:", len(all_examples))
print("Unique passages:", len(set(ex["passage_id"] for ex in all_examples)))
print("Languages:", set(ex["language"] for ex in all_examples))

Total examples: 14
Unique passages: 4
Languages: {'en'}


In [9]:
english_examples = [
    # THE LAMP ON HARROW POINT
    {
        "passage_id": "lamp_01",
        "language": "en",
        "context": """Every evening at dusk, Mara climbed the ninety-one steps of the lighthouse to light the lamp, just as her grandfather had done before her. The lamp itself was old, but it never failed. What worried her was the harbor below, where fewer boats returned each season.

Tonight, a single fishing boat wobbled toward the rocks. Mara did not wave or shout. She turned the lamp's shutter slowly, three long flashes and one short, the way her grandfather had taught her when she was small. The boat straightened and slid into the channel between the stones.

At the dock, the fisherman looked up at the tower and lifted his cap. He had never met Mara, yet he knew exactly who was up there. The old signal had told him that the person keeping the light still remembered the old ways, and that the channel was safe to trust.""",
        "answer": "ninety-one steps",
        "question": "How many steps did Mara climb to reach the lighthouse lamp?"
    },
    {
        "passage_id": "lamp_01",
        "language": "en",
        "context": """Every evening at dusk, Mara climbed the ninety-one steps of the lighthouse to light the lamp, just as her grandfather had done before her. The lamp itself was old, but it never failed. What worried her was the harbor below, where fewer boats returned each season.

Tonight, a single fishing boat wobbled toward the rocks. Mara did not wave or shout. She turned the lamp's shutter slowly, three long flashes and one short, the way her grandfather had taught her when she was small. The boat straightened and slid into the channel between the stones.

At the dock, the fisherman looked up at the tower and lifted his cap. He had never met Mara, yet he knew exactly who was up there. The old signal had told him that the person keeping the light still remembered the old ways, and that the channel was safe to trust.""",
        "answer": "three long flashes and one short",
        "question": "What pattern of flashes did Mara use to guide the fishing boat?"
    },
    {
        "passage_id": "lamp_01",
        "language": "en",
        "context": """Every evening at dusk, Mara climbed the ninety-one steps of the lighthouse to light the lamp, just as her grandfather had done before her. The lamp itself was old, but it never failed. What worried her was the harbor below, where fewer boats returned each season.

Tonight, a single fishing boat wobbled toward the rocks. Mara did not wave or shout. She turned the lamp's shutter slowly, three long flashes and one short, the way her grandfather had taught her when she was small. The boat straightened and slid into the channel between the stones.

At the dock, the fisherman looked up at the tower and lifted his cap. He had never met Mara, yet he knew exactly who was up there. The old signal had told him that the person keeping the light still remembered the old ways, and that the channel was safe to trust.""",
        "answer": "her grandfather had taught her when she was small",
        "question": "Who taught Mara the lighthouse signal when she was young?"
    },
    {
        "passage_id": "lamp_01",
        "language": "en",
        "context": """Every evening at dusk, Mara climbed the ninety-one steps of the lighthouse to light the lamp, just as her grandfather had done before her. The lamp itself was old, but it never failed. What worried her was the harbor below, where fewer boats returned each season.

Tonight, a single fishing boat wobbled toward the rocks. Mara did not wave or shout. She turned the lamp's shutter slowly, three long flashes and one short, the way her grandfather had taught her when she was small. The boat straightened and slid into the channel between the stones.

At the dock, the fisherman looked up at the tower and lifted his cap. He had never met Mara, yet he knew exactly who was up there. The old signal had told him that the person keeping the light still remembered the old ways, and that the channel was safe to trust.""",
        "answer": "the channel was safe to trust",
        "question": "What did the old signal communicate to the fisherman about the channel?"
    },

    # THE LAST JAR OF SEEDS
    {
        "passage_id": "seeds_01",
        "language": "en",
        "context": """When the rains failed for the second year, the community garden shrank to a single raised bed behind the old schoolhouse. Teodora kept one jar of bean seeds on the highest shelf in her kitchen, untouched, while the rest of the garden's seeds went into the ground early, before anyone was sure the soil could hold them.

Her neighbors asked why she didn't plant her share like everyone else. She only said that a garden needed one thing kept in reserve, in case the first planting failed entirely. When a late frost killed half the seedlings in May, it was her jar, and no one else's, that refilled the empty rows by the second week of June.

Nobody suggested she explain herself again after that. The following spring, three other families on her street had jars of their own on their highest shelves, untouched, waiting.""",
        "answer": "a single raised bed",
        "question": "What remained of the community garden after the rains failed for the second year?"
    },
    {
        "passage_id": "seeds_01",
        "language": "en",
        "context": """When the rains failed for the second year, the community garden shrank to a single raised bed behind the old schoolhouse. Teodora kept one jar of bean seeds on the highest shelf in her kitchen, untouched, while the rest of the garden's seeds went into the ground early, before anyone was sure the soil could hold them.

Her neighbors asked why she didn't plant her share like everyone else. She only said that a garden needed one thing kept in reserve, in case the first planting failed entirely. When a late frost killed half the seedlings in May, it was her jar, and no one else's, that refilled the empty rows by the second week of June.

Nobody suggested she explain herself again after that. The following spring, three other families on her street had jars of their own on their highest shelves, untouched, waiting.""",
        "answer": "one jar of bean seeds",
        "question": "What did Teodora keep untouched on the highest shelf in her kitchen?"
    },
    {
        "passage_id": "seeds_01",
        "language": "en",
        "context": """When the rains failed for the second year, the community garden shrank to a single raised bed behind the old schoolhouse. Teodora kept one jar of bean seeds on the highest shelf in her kitchen, untouched, while the rest of the garden's seeds went into the ground early, before anyone was sure the soil could hold them.

Her neighbors asked why she didn't plant her share like everyone else. She only said that a garden needed one thing kept in reserve, in case the first planting failed entirely. When a late frost killed half the seedlings in May, it was her jar, and no one else's, that refilled the empty rows by the second week of June.

Nobody suggested she explain herself again after that. The following spring, three other families on her street had jars of their own on their highest shelves, untouched, waiting.""",
        "answer": "a late frost killed half the seedlings in May",
        "question": "What event made Teodora's reserved seeds necessary?"
    },
    {
        "passage_id": "seeds_01",
        "language": "en",
        "context": """When the rains failed for the second year, the community garden shrank to a single raised bed behind the old schoolhouse. Teodora kept one jar of bean seeds on the highest shelf in her kitchen, untouched, while the rest of the garden's seeds went into the ground early, before anyone was sure the soil could hold them.

Her neighbors asked why she didn't plant her share like everyone else. She only said that a garden needed one thing kept in reserve, in case the first planting failed entirely. When a late frost killed half the seedlings in May, it was her jar, and no one else's, that refilled the empty rows by the second week of June.

Nobody suggested she explain herself again after that. The following spring, three other families on her street had jars of their own on their highest shelves, untouched, waiting.""",
        "answer": "three other families on her street had jars of their own",
        "question": "How did Teodora's neighbors respond to her decision to save seeds?"
    },

    # THE QUIET MACHINE
    {
        "passage_id": "machine_01",
        "language": "en",
        "context": """The record player had sat in the back room of the library for eleven years, donated along with a box of records nobody had catalogued. When Priya finally opened the cabinet to clear space, she found a needle worn down to almost nothing and a handwritten label taped inside the lid: "Return to Mr. Oyelaran when fixed."

She looked him up before she decided what to do with any of it. Mr. Oyelaran had taught music at the school across the street until it closed in the nineties; the library had no record of him after that. Still, she ordered a new needle, cleaned the turntable herself over a weekend, and played the first record in the box just to test it.

It was a recording of a school choir, slightly out of tune, dated the same year the label was written. Priya didn't know whose voice was whose. She reshelved the player in the front room anyway, with a handwritten card of her own: "Ask at the desk to listen." """,
        "answer": "eleven years",
        "question": "How long had the record player been stored in the back room of the library?"
    },
    {
        "passage_id": "machine_01",
        "language": "en",
        "context": """The record player had sat in the back room of the library for eleven years, donated along with a box of records nobody had catalogued. When Priya finally opened the cabinet to clear space, she found a needle worn down to almost nothing and a handwritten label taped inside the lid: "Return to Mr. Oyelaran when fixed."

She looked him up before she decided what to do with any of it. Mr. Oyelaran had taught music at the school across the street until it closed in the nineties; the library had no record of him after that. Still, she ordered a new needle, cleaned the turntable herself over a weekend, and played the first record in the box just to test it.

It was a recording of a school choir, slightly out of tune, dated the same year the label was written. Priya didn't know whose voice was whose. She reshelved the player in the front room anyway, with a handwritten card of her own: "Ask at the desk to listen." """,
        "answer": "Return to Mr. Oyelaran when fixed",
        "question": "What instruction was written on the label inside the record player's lid?"
    },
    {
        "passage_id": "machine_01",
        "language": "en",
        "context": """The record player had sat in the back room of the library for eleven years, donated along with a box of records nobody had catalogued. When Priya finally opened the cabinet to clear space, she found a needle worn down to almost nothing and a handwritten label taped inside the lid: "Return to Mr. Oyelaran when fixed."

She looked him up before she decided what to do with any of it. Mr. Oyelaran had taught music at the school across the street until it closed in the nineties; the library had no record of him after that. Still, she ordered a new needle, cleaned the turntable herself over a weekend, and played the first record in the box just to test it.

It was a recording of a school choir, slightly out of tune, dated the same year the label was written. Priya didn't know whose voice was whose. She reshelved the player in the front room anyway, with a handwritten card of her own: "Ask at the desk to listen." """,
        "answer": "ordered a new needle, cleaned the turntable herself over a weekend",
        "question": "How did Priya repair the record player?"
    },
    {
        "passage_id": "machine_01",
        "language": "en",
        "context": """The record player had sat in the back room of the library for eleven years, donated along with a box of records nobody had catalogued. When Priya finally opened the cabinet to clear space, she found a needle worn down to almost nothing and a handwritten label taped inside the lid: "Return to Mr. Oyelaran when fixed."

She looked him up before she decided what to do with any of it. Mr. Oyelaran had taught music at the school across the street until it closed in the nineties; the library had no record of him after that. Still, she ordered a new needle, cleaned the turntable herself over a weekend, and played the first record in the box just to test it.

It was a recording of a school choir, slightly out of tune, dated the same year the label was written. Priya didn't know whose voice was whose. She reshelved the player in the front room anyway, with a handwritten card of her own: "Ask at the desk to listen." """,
        "answer": "a recording of a school choir",
        "question": "What did Priya discover on the first record she played?"
    }
]

print("English examples:", len(english_examples))
print("Unique passages:", len(set(ex["passage_id"] for ex in english_examples)))

English examples: 12
Unique passages: 3


In [18]:
# 86 new original English examples for T5 question generation.
# Keep your existing `all_examples` unchanged.

passages = [
    {
        "passage_id": "science_01",
        "context": (
            "Plants use sunlight, water, and carbon dioxide to make their own food. "
            "This process is called photosynthesis. Chlorophyll, the green pigment "
            "in leaves, absorbs the light needed for this process."
        ),
        "qa": [
            ("sunlight, water, and carbon dioxide", "What materials and energy source do plants use to make food?"),
            ("photosynthesis", "What is the process called in which plants make their own food?"),
            ("Chlorophyll", "What green pigment absorbs light in plant leaves?"),
            ("absorbs the light needed for this process", "What does chlorophyll do during photosynthesis?")
        ]
    },
    {
        "passage_id": "science_02",
        "context": (
            "The water cycle moves water between Earth's surface and the atmosphere. "
            "Heat from the Sun causes liquid water to evaporate into water vapor. "
            "The vapor cools and forms clouds through a process called condensation."
        ),
        "qa": [
            ("the atmosphere", "Where does water move as part of the water cycle?"),
            ("Heat from the Sun", "What causes liquid water to evaporate?"),
            ("water vapor", "What does liquid water become during evaporation?"),
            ("condensation", "What process forms clouds when water vapor cools?")
        ]
    },
    {
        "passage_id": "science_03",
        "context": (
            "Earth has several layers, including the crust, mantle, and core. "
            "The crust is the thin outer layer where people live. Beneath it, "
            "the mantle contains very hot rock that moves slowly over time."
        ),
        "qa": [
            ("the crust, mantle, and core", "What are three major layers of Earth?"),
            ("the thin outer layer", "What kind of layer is Earth's crust?"),
            ("where people live", "Where do people live in relation to Earth's layers?"),
            ("moves slowly over time", "How does the hot rock in the mantle behave?")
        ]
    },
    {
        "passage_id": "science_04",
        "context": (
            "Sound begins when an object vibrates. These vibrations travel through "
            "a medium such as air, water, or a solid object. In space, where there "
            "is almost no matter, ordinary sound cannot travel in the same way."
        ),
        "qa": [
            ("an object vibrates", "What causes sound to begin?"),
            ("air, water, or a solid object", "What are three examples of mediums through which sound can travel?"),
            ("a medium", "What must sound vibrations travel through?"),
            ("there is almost no matter", "Why can ordinary sound not travel through space in the same way?")
        ]
    },
    {
        "passage_id": "nature_01",
        "context": (
            "Mangrove trees grow along many tropical coastlines. Their tangled roots "
            "help hold soil in place and reduce the force of waves. These roots also "
            "provide shelter for young fish and other small marine animals."
        ),
        "qa": [
            ("along many tropical coastlines", "Where do mangrove trees grow?"),
            ("hold soil in place", "How do mangrove roots help protect coastal land?"),
            ("reduce the force of waves", "What effect do mangrove roots have on waves?"),
            ("young fish and other small marine animals", "Which animals find shelter among mangrove roots?")
        ]
    },
    {
        "passage_id": "nature_02",
        "context": (
            "Bees visit flowers to collect nectar and pollen. As bees move from one "
            "flower to another, pollen can stick to their bodies and reach other "
            "flowers. This movement helps many plants produce seeds and fruit."
        ),
        "qa": [
            ("nectar and pollen", "What do bees collect from flowers?"),
            ("stick to their bodies", "How can pollen travel with a bee?"),
            ("one flower to another", "Where do bees move while carrying pollen?"),
            ("produce seeds and fruit", "What can pollination help plants do?")
        ]
    },
    {
        "passage_id": "nature_03",
        "context": (
            "Bats often rest in caves, hollow trees, or dark spaces during the day. "
            "Many species become active at night to search for food. Some bats eat "
            "insects, while others feed on fruit or nectar."
        ),
        "qa": [
            ("caves, hollow trees, or dark spaces", "Where might bats rest during the day?"),
            ("at night", "When do many bat species become active to search for food?"),
            ("insects", "What do some bats eat?"),
            ("fruit or nectar", "What do some bats eat instead of insects?")
        ]
    },
    {
        "passage_id": "nature_04",
        "context": (
            "Coral reefs are built by tiny animals called coral polyps. Over time, "
            "their hard skeletons form structures that provide shelter for many sea "
            "creatures. Pollution and unusually warm ocean water can damage reefs."
        ),
        "qa": [
            ("coral polyps", "What tiny animals build coral reefs?"),
            ("their hard skeletons", "What forms the structures of coral reefs over time?"),
            ("shelter for many sea creatures", "What do coral reef structures provide?"),
            ("unusually warm ocean water", "What kind of ocean condition can damage coral reefs?")
        ]
    },
    {
        "passage_id": "history_01",
        "context": (
            "Ancient communities often formed near rivers because rivers supplied "
            "water for drinking and farming. Rivers also provided routes for travel "
            "and trade. However, communities had to prepare for floods that could "
            "damage homes and crops."
        ),
        "qa": [
            ("near rivers", "Where did many ancient communities form?"),
            ("water for drinking and farming", "Why were rivers important for daily life?"),
            ("travel and trade", "What activities did rivers help people carry out?"),
            ("floods", "What natural event could damage homes and crops near rivers?")
        ]
    },
    {
        "passage_id": "history_02",
        "context": (
            "The printing press made it possible to produce many copies of a text "
            "more quickly than copying each page by hand. As printed books became "
            "more available, more people gained access to written information. "
            "This helped ideas spread across different regions."
        ),
        "qa": [
            ("many copies of a text", "What could the printing press produce?"),
            ("copying each page by hand", "What method was slower than using the printing press?"),
            ("written information", "What did more people gain access to as printed books became available?"),
            ("ideas spread across different regions", "What happened to ideas as printed materials circulated?")
        ]
    },
    {
        "passage_id": "history_03",
        "context": (
            "Early navigators used stars to help determine direction during night "
            "journeys. They also observed the Sun during the day and studied coastlines "
            "and landmarks. These observations helped travelers plan routes before "
            "modern satellite navigation existed."
        ),
        "qa": [
            ("stars", "What did early navigators observe to help determine direction at night?"),
            ("the Sun", "What did navigators observe during the day?"),
            ("coastlines and landmarks", "What features did navigators study along their routes?"),
            ("plan routes", "How did these observations help travelers?")
        ]
    },
    {
        "passage_id": "history_04",
        "context": (
            "Public libraries allow people to borrow books and use learning resources. "
            "Some libraries also offer computers, workshops, and quiet places to study. "
            "By providing these services, libraries support education in their communities."
        ),
        "qa": [
            ("books", "What can people borrow from public libraries?"),
            ("computers, workshops, and quiet places to study", "What additional resources or services do some libraries offer?"),
            ("education", "What do libraries support in their communities?"),
            ("their communities", "Whose education do public libraries help support?")
        ]
    },
    {
        "passage_id": "daily_01",
        "context": (
            "Mika placed a reusable water bottle in her school bag each morning. "
            "She refilled it at the water station instead of buying drinks in plastic "
            "bottles. After several weeks, she noticed that she had thrown away fewer "
            "plastic containers."
        ),
        "qa": [
            ("a reusable water bottle", "What did Mika place in her school bag each morning?"),
            ("at the water station", "Where did Mika refill her bottle?"),
            ("plastic bottles", "What kind of containers did Mika avoid buying drinks in?"),
            ("fewer plastic containers", "What change did Mika notice after several weeks?")
        ]
    },
    {
        "passage_id": "daily_02",
        "context": (
            "Before an examination, Luis divided his study time into short sessions. "
            "He reviewed one topic, took a brief break, and then moved to the next topic. "
            "This routine helped him notice which lessons needed more attention."
        ),
        "qa": [
            ("short sessions", "How did Luis divide his study time?"),
            ("one topic", "What did Luis review before taking a break?"),
            ("a brief break", "What did Luis take between study sessions?"),
            ("which lessons needed more attention", "What did the routine help Luis identify?")
        ]
    },
    {
        "passage_id": "daily_03",
        "context": (
            "Nina checked the weather forecast before leaving for school. The forecast "
            "predicted heavy rain in the afternoon, so she packed a folding umbrella. "
            "When the rain began later, she could walk home without getting soaked."
        ),
        "qa": [
            ("the weather forecast", "What did Nina check before leaving for school?"),
            ("heavy rain in the afternoon", "What weather did the forecast predict?"),
            ("a folding umbrella", "What did Nina pack because of the forecast?"),
            ("without getting soaked", "How did the umbrella help Nina walk home?")
        ]
    },
    {
        "passage_id": "daily_04",
        "context": (
            "A neighborhood group organized a weekend cleanup beside a small creek. "
            "Volunteers collected wrappers, cans, and other litter from the banks. "
            "They separated recyclable materials from waste before taking the bags "
            "to the collection point."
        ),
        "qa": [
            ("a weekend cleanup", "What activity did the neighborhood group organize?"),
            ("wrappers, cans, and other litter", "What did volunteers collect from the creek banks?"),
            ("recyclable materials from waste", "What did volunteers separate before leaving?"),
            ("the collection point", "Where did volunteers take the bags?")
        ]
    },
    {
        "passage_id": "inference_01",
        "context": (
            "When Carlo entered the classroom, he saw wet footprints leading to the "
            "window. His open notebook had several drops of water on its pages, and "
            "the window was still partly open. He quickly moved his notebook to a dry desk."
        ),
        "qa": [
            ("wet footprints", "What sign of moisture did Carlo notice on the classroom floor?"),
            ("several drops of water", "What was on the pages of Carlo's notebook?"),
            ("partly open", "What was the position of the window?"),
            ("a dry desk", "Where did Carlo move his notebook?")
        ]
    },
    {
        "passage_id": "inference_02",
        "context": (
            "Amara practiced the same piano passage every evening. At first, she stopped "
            "several times to correct mistakes. After two weeks, she played the passage "
            "with fewer pauses and needed to check the music sheet less often."
        ),
        "qa": [
            ("every evening", "When did Amara practice the piano passage?"),
            ("several times", "How often did Amara stop at first to correct mistakes?"),
            ("two weeks", "How long did Amara practice before showing improvement?"),
            ("fewer pauses", "What change showed that Amara was improving?")
        ]
    },
    {
        "passage_id": "inference_03",
        "context": (
            "The hallway grew quiet when the principal entered. Several students "
            "straightened their uniforms, and one student quickly put away a phone. "
            "The principal looked around the hallway before greeting the students."
        ),
        "qa": [
            ("the principal", "Who entered the hallway?"),
            ("straightened their uniforms", "What did several students do when the principal arrived?"),
            ("put away a phone", "What did one student do quickly?"),
            ("greeting the students", "What did the principal do after looking around?")
        ]
    },
    {
        "passage_id": "inference_04",
        "context": (
            "The bakery usually sold out of bread by noon. On Tuesday, the shelves "
            "were still full late in the afternoon, and few customers had entered "
            "the shop. The baker decided to prepare a smaller batch the next morning."
        ),
        "qa": [
            ("by noon", "When did the bakery usually sell out of bread?"),
            ("late in the afternoon", "When were the shelves still full on Tuesday?"),
            ("few customers", "How many customers had entered the shop on Tuesday?"),
            ("a smaller batch", "What did the baker decide to prepare the next morning?")
        ]
    },
    {
        "passage_id": "problem_01",
        "context": (
            "A school garden's soil became dry during a week without rain. The students "
            "spread mulch around the plants to slow water loss from the soil. They also "
            "made a watering schedule so each garden bed received enough water."
        ),
        "qa": [
            ("a week without rain", "What caused the school garden's soil to become dry?"),
            ("mulch", "What did the students spread around the plants?"),
            ("slow water loss from the soil", "Why did the students use mulch?"),
            ("a watering schedule", "What did the students make to ensure each bed received water?")
        ]
    },
    {
        "passage_id": "problem_02",
        "context": (
            "During a group project, Bea noticed that two classmates were completing "
            "the same task while another task remained unfinished. She suggested listing "
            "all the remaining jobs and assigning one job to each available member. "
            "The group then completed the project before the deadline."
        ),
        "qa": [
            ("the same task", "What were two classmates completing?"),
            ("listing all the remaining jobs", "What did Bea suggest the group do first?"),
            ("one job to each available member", "How did Bea suggest dividing the work?"),
            ("before the deadline", "When did the group complete the project?")
        ]
    },
    {
        "passage_id": "problem_03",
        "context": (
            "A bicycle's front tire kept losing air, even after it was pumped up. "
            "Ramon checked the tire and found a small piece of glass stuck in the rubber. "
            "He removed the glass and patched the hole before inflating the tire again."
        ),
        "qa": [
            ("front tire", "Which part of the bicycle kept losing air?"),
            ("a small piece of glass", "What did Ramon find stuck in the rubber?"),
            ("patched the hole", "What did Ramon do after removing the glass?"),
            ("inflating the tire again", "What did Ramon do after patching the hole?")
        ]
    },
    {
        "passage_id": "environment_01",
        "context": (
            "A town installed rain barrels beside several public buildings. The barrels "
            "collected rainwater from roof gutters, and workers later used the stored "
            "water to irrigate flower beds. This reduced the amount of treated tap water "
            "needed for the gardens."
        ),
        "qa": [
            ("rain barrels", "What did the town install beside public buildings?"),
            ("roof gutters", "Where did the rainwater enter the barrels from?"),
            ("irrigate flower beds", "How did workers use the stored rainwater?"),
            ("treated tap water", "What type of water did the gardens need less of?")
        ]
    },
    {
        "passage_id": "environment_02",
        "context": (
            "Sea turtles lay their eggs in nests dug into sandy beaches. Artificial "
            "lights near the shore can confuse hatchlings that normally move toward "
            "the brighter natural horizon over the sea. Some coastal communities reduce "
            "this problem by shielding lights during nesting season."
        ),
        "qa": [
            ("sandy beaches", "Where do sea turtles lay their eggs?"),
            ("hatchlings", "Which young animals can be confused by artificial coastal lights?"),
            ("the brighter natural horizon over the sea", "Toward what do hatchlings normally move?"),
            ("shielding lights during nesting season", "How do some coastal communities reduce the problem?")
        ]
    },
    {
        "passage_id": "environment_03",
        "context": (
            "Food scraps and dry leaves can be placed in a compost pile. Tiny organisms "
            "break down this material over time and turn it into a dark, soil-like "
            "substance. Gardeners mix the finished compost into soil to add organic matter."
        ),
        "qa": [
            ("Food scraps and dry leaves", "What materials can be placed in a compost pile?"),
            ("Tiny organisms", "What breaks down the materials in a compost pile?"),
            ("a dark, soil-like substance", "What does the material become over time?"),
            ("add organic matter", "Why do gardeners mix finished compost into soil?")
        ]
    },
    {
        "passage_id": "environment_04",
        "context": (
            "Some cities plant trees along streets to provide shade for pedestrians. "
            "Tree canopies can also help cool nearby surfaces on hot days. To keep street "
            "trees healthy, workers must provide enough space for roots and protect them "
            "from damage."
        ),
        "qa": [
            ("shade for pedestrians", "What do street trees provide for people walking?"),
            ("cool nearby surfaces", "How can tree canopies help on hot days?"),
            ("enough space for roots", "What must workers provide to keep street trees healthy?"),
            ("protect them from damage", "What else must workers do for street trees?")
        ]
    },
    {
        "passage_id": "technology_01",
        "context": (
            "A computer stores information in files that can be organized into folders. "
            "Clear file names help users find documents later. Making a backup on another "
            "device can protect important work if the original computer stops working."
        ),
        "qa": [
            ("files", "What does a computer use to store information?"),
            ("folders", "Where can files be organized?"),
            ("Clear file names", "What helps users find documents later?"),
            ("another device", "Where can a backup be stored to protect important work?")
        ]
    },
    {
        "passage_id": "technology_02",
        "context": (
            "A solar panel converts energy from sunlight into electrical energy. "
            "The electricity can power devices immediately or charge a battery for later use. "
            "The amount of electricity produced depends partly on sunlight reaching the panel."
        ),
        "qa": [
            ("electrical energy", "What kind of energy does a solar panel produce?"),
            ("a battery", "What can store electricity for later use?"),
            ("sunlight reaching the panel", "What partly affects how much electricity is produced?"),
            ("power devices immediately", "What can the electricity from a solar panel do without being stored?")
        ]
    },
    {
        "passage_id": "technology_03",
        "context": (
            "A strong password is difficult for other people to guess. Using a different "
            "password for each important account limits the damage if one password is exposed. "
            "A password manager can help users store unique passwords securely."
        ),
        "qa": [
            ("difficult for other people to guess", "What should a strong password be like?"),
            ("a different password for each important account", "What password habit can limit the damage from an exposed password?"),
            ("one password is exposed", "What situation can make unique passwords especially useful?"),
            ("a password manager", "What tool can help users store unique passwords securely?")
        ]
    },
    {
        "passage_id": "technology_04",
        "context": (
            "A robot uses sensors to collect information about its surroundings. Its "
            "controller processes that information and sends instructions to its motors. "
            "Together, these parts allow the robot to respond to obstacles and follow a route."
        ),
        "qa": [
            ("sensors", "What does a robot use to collect information about its surroundings?"),
            ("its controller", "What processes the information collected by the sensors?"),
            ("its motors", "Where does the controller send instructions?"),
            ("respond to obstacles and follow a route", "What can the robot do with its connected parts?")
        ]
    },
    {
        "passage_id": "health_01",
        "context": (
            "Sleep gives the body time to rest and recover. Students who keep a regular "
            "sleep schedule may find it easier to feel alert during the day. Bright screens "
            "and stimulating activities close to bedtime can make it harder for some people "
            "to fall asleep."
        ),
        "qa": [
            ("rest and recover", "What does sleep give the body time to do?"),
            ("a regular sleep schedule", "What habit may help students feel alert during the day?"),
            ("Bright screens and stimulating activities", "What can make falling asleep harder close to bedtime?"),
            ("during the day", "When may students find it easier to feel alert with a regular sleep schedule?")
        ]
    },
    {
        "passage_id": "health_02",
        "context": (
            "The human heart pumps blood through a network of blood vessels. Blood carries "
            "oxygen and nutrients to cells throughout the body. It also transports some waste "
            "products to organs that help remove them."
        ),
        "qa": [
            ("blood", "What does the human heart pump through blood vessels?"),
            ("a network of blood vessels", "Through what network does the heart pump blood?"),
            ("oxygen and nutrients", "What does blood carry to cells?"),
            ("organs that help remove them", "Where are some waste products transported for removal?")
        ]
    },
    {
        "passage_id": "health_03",
        "context": (
            "Regular physical activity strengthens muscles and can support heart health. "
            "Walking, cycling, dancing, and playing sports are examples of movement. "
            "Choosing an enjoyable activity can make it easier to continue the habit."
        ),
        "qa": [
            ("muscles", "What can regular physical activity strengthen?"),
            ("heart health", "What aspect of health can physical activity support?"),
            ("Walking, cycling, dancing, and playing sports", "What are examples of physical movement?"),
            ("an enjoyable activity", "What kind of activity may be easier to continue regularly?")
        ]
    },
    {
        "passage_id": "community_01",
        "context": (
            "The residents of Maple Street wanted a safer crossing near their school. "
            "They recorded the times when traffic was busiest and presented the information "
            "to the local council. After reviewing the report, the council arranged for "
            "a crossing guard to be present during school arrival and dismissal."
        ),
        "qa": [
            ("a safer crossing", "What did the residents of Maple Street want near their school?"),
            ("the times when traffic was busiest", "What information did the residents record?"),
            ("the local council", "To whom did the residents present their information?"),
            ("a crossing guard", "What did the council arrange to help students cross safely?")
        ]
    },
    {
        "passage_id": "community_02",
        "context": (
            "A community center started a book exchange for families. Visitors could bring "
            "books they had finished reading and choose other books to take home. Within a "
            "month, the shelves held stories, science books, and picture books in several "
            "languages."
        ),
        "qa": [
            ("a book exchange", "What activity did the community center start?"),
            ("books they had finished reading", "What could visitors bring to the exchange?"),
            ("other books to take home", "What could visitors choose after bringing their books?"),
            ("several languages", "In how many kinds of languages were the books available?")
        ]
    }
]

# Build 86 new examples: 21 passages x 4 examples, plus 2 from the final passage.
new_examples = []

for passage in passages:
    for answer, question in passage["qa"]:
        new_examples.append({
            "passage_id": passage["passage_id"],
            "language": "en",
            "context": passage["context"],
            "answer": answer,
            "question": question
        })

# Select 86 examples across the new passages.
# Take 4 from the first 21 passages, and 2 from the last passage.
new_examples = [
    ex
    for passage in passages
    for ex in [
        {
            "passage_id": passage["passage_id"],
            "language": "en",
            "context": passage["context"],
            "answer": answer,
            "question": question
        }
        for answer, question in passage["qa"]
    ][:4]
]

# Keep 4 examples from each of the first 21 passages, and 2 from the final passage.
# There are 31 passages in this cell, so choose 21 full passages plus the next half.
new_examples = [
    ex
    for passage in passages[:21]
    for answer, question in passage["qa"]
    for ex in [{
        "passage_id": passage["passage_id"],
        "language": "en",
        "context": passage["context"],
        "answer": answer,
        "question": question
    }]
]

# Add two examples from one additional passage.
passage = passages[21]
for answer, question in passage["qa"][:2]:
    new_examples.append({
        "passage_id": passage["passage_id"],
        "language": "en",
        "context": passage["context"],
        "answer": answer,
        "question": question
    })

# Combine with the 14 examples already prepared.
expanded_examples = all_examples + new_examples

print("Existing examples:", len(all_examples))
print("New examples:", len(new_examples))
print("Combined examples:", len(expanded_examples))
print("Unique passages:", len(set(ex["passage_id"] for ex in expanded_examples)))

Existing examples: 14
New examples: 86
Combined examples: 100
Unique passages: 26


#**checkpoint**

In [22]:
import json
from google.colab import files

with open("reading_comprehension_dataset.json", "w", encoding="utf-8") as f:
    json.dump(expanded_examples, f, ensure_ascii=False, indent=2)

files.download("reading_comprehension_dataset.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

# Group by passage to prevent data leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        expanded_examples,
        groups=[ex["passage_id"] for ex in expanded_examples]
    )
)

train_examples = [expanded_examples[i] for i in train_idx]
val_examples = [expanded_examples[i] for i in val_idx]

train_passages = set(ex["passage_id"] for ex in train_examples)
val_passages = set(ex["passage_id"] for ex in val_examples)

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Training passages:", len(train_passages))
print("Validation passages:", len(val_passages))
print("Passages overlap:", train_passages & val_passages)

Training examples: 76
Validation examples: 24
Training passages: 20
Validation passages: 6
Passages overlap: set()


In [25]:
def prepare_t5_example(example):
    context = example["context"]
    answer = example["answer"]

    # Highlight the exact answer span
    start = context.find(answer)

    if start == -1:
        raise ValueError(
            f"Answer not found in passage: {example['passage_id']} - {answer}"
        )

    # Make sure the answer occurs only once
    if context.count(answer) != 1:
        raise ValueError(
            f"Answer occurs {context.count(answer)} times: "
            f"{example['passage_id']} - {answer}"
        )

    highlighted_context = (
        context[:start]
        + "<hl> "
        + answer
        + " <hl>"
        + context[start + len(answer):]
    )

    return {
        "input_text": highlighted_context,
        "target_text": example["question"],
        "passage_id": example["passage_id"]
    }


train_data = [prepare_t5_example(ex) for ex in train_examples]
val_data = [prepare_t5_example(ex) for ex in val_examples]

print("Training records:", len(train_data))
print("Validation records:", len(val_data))
print("\nSample input:")
print(train_data[0]["input_text"])
print("\nExpected question:")
print(train_data[0]["target_text"])

Training records: 76
Validation records: 24

Sample input:
Daniel visited the community garden every Saturday. He helped move the plants near the fence where they received <hl> more sunlight <hl>.

Expected question:
What did the plants receive near the fence?


In [26]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "valhalla/t5-base-qg-hl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

def tokenize_function(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=384,
        truncation=True
    )

    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=val_dataset.column_names
)

print("Tokenized training examples:", len(tokenized_train))
print("Tokenized validation examples:", len(tokenized_val))
print("Tokenizer:", MODEL_NAME)

Map:   0%|          | 0/76 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenized training examples: 76
Tokenized validation examples: 24
Tokenizer: valhalla/t5-base-qg-hl


In [27]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable a GPU in Runtime > Change runtime type.")

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [28]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

MODEL_NAME = "valhalla/t5-base-qg-hl"
OUTPUT_DIR = "/content/t5-qg-finetuned"

# Load the pretrained question-generation model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Prepare batches with padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training settings
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluate and save after every epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Use your Tesla T4 GPU
    fp16=torch.cuda.is_available(),

    logging_steps=5,
    report_to="none",
    seed=42
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Model:", MODEL_NAME)
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Epochs:", training_args.num_train_epochs)
print("GPU:", torch.cuda.get_device_name(0))

Model: valhalla/t5-base-qg-hl
Training examples: 76
Validation examples: 24
Epochs: 5
GPU: Tesla T4


In [29]:
train_result = trainer.train()

print("\nTraining completed!")
print("Training loss:", train_result.training_loss)

print("\nFinal validation results:")
print(trainer.evaluate())

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,1.218900,0.907070
2,1.047700,0.879946
3,1.042400,0.868772
4,0.959500,0.860517
5,1.066600,0.857862


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training completed!
Training loss: 0.99356891230533

Final validation results:


{'eval_loss': 0.8578617572784424, 'eval_runtime': 0.2842, 'eval_samples_per_second': 84.457, 'eval_steps_per_second': 21.114, 'epoch': 5.0}


In [30]:
SAVE_DIR = "/content/t5-qg-finetuned-final"

# Save the best model selected by validation loss
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model and tokenizer saved to:", SAVE_DIR)

Model and tokenizer saved to: /content/t5-qg-finetuned-final


In [31]:
import shutil

shutil.make_archive(
    "/content/t5-qg-finetuned-final",
    "zip",
    "/content/t5-qg-finetuned-final"
)

from google.colab import files
files.download("/content/t5-qg-finetuned-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_MODEL = "valhalla/t5-base-qg-hl"
FINETUNED_MODEL = "/content/t5-qg-finetuned-final"

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to("cuda")
base_model.eval()

fine_tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL)
fine_model = AutoModelForSeq2SeqLM.from_pretrained(FINETUNED_MODEL).to("cuda")
fine_model.eval()

test_context = (
    "A group of students built a small weather station behind their school. "
    "They recorded rainfall each morning and measured temperature at noon. "
    "After comparing the records for a month, they noticed that rainy days "
    "were often cooler than sunny days."
)

test_answer = "each morning"

start = test_context.find(test_answer)
assert start != -1 and test_context.count(test_answer) == 1

test_input = (
    test_context[:start]
    + "<hl> "
    + test_answer
    + " <hl>"
    + test_context[start + len(test_answer):]
)

def generate_question(model, tokenizer, text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Answer:", test_answer)
print("\nOriginal T5:")
print(generate_question(base_model, base_tokenizer, test_input))

print("\nFine-tuned T5:")
print(generate_question(fine_model, fine_tokenizer, test_input))

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Answer: each morning

Original T5:
How often did the students record rainfall?

Fine-tuned T5:
When did the students record rainfall?


In [33]:
test_cases = [
    {
        "context": (
            "Mia planted tomato seeds in a small pot. "
            "She watered them every afternoon and placed the pot near a sunny window. "
            "After two weeks, tiny green leaves appeared."
        ),
        "answer": "After two weeks"
    },
    {
        "context": (
            "The school library introduced a reading program. "
            "Students who read three books each month could earn certificates. "
            "The program encouraged more students to visit the library."
        ),
        "answer": "three books each month"
    },
    {
        "context": (
            "Heavy rain flooded several streets in the town. "
            "Local volunteers helped residents move their belongings to safer places. "
            "The town council later organized a cleanup."
        ),
        "answer": "Local volunteers"
    },
]

for i, case in enumerate(test_cases, start=1):
    context = case["context"]
    answer = case["answer"]

    start = context.find(answer)
    assert start != -1 and context.count(answer) == 1

    highlighted = (
        context[:start] + "<hl> " + answer + " <hl>"
        + context[start + len(answer):]
    )

    print(f"\n{'=' * 45}")
    print(f"TEST {i}")
    print("Answer:", answer)
    print("Original T5:")
    print(generate_question(base_model, base_tokenizer, highlighted))
    print("Fine-tuned T5:")
    print(generate_question(fine_model, fine_tokenizer, highlighted))


TEST 1
Answer: After two weeks
Original T5:
How long did it take for tomatoes to turn green?
Fine-tuned T5:
When did tiny green leaves appear on the tomato seeds?

TEST 2
Answer: three books each month
Original T5:
How many books did students read each month?
Fine-tuned T5:
How many books did students read each month?

TEST 3
Answer: Local volunteers
Original T5:
Who helped residents move their belongings to safer places?
Fine-tuned T5:
Who helped residents move their belongings?


In [34]:
test_cases = [
    {
        "context": (
            "Daniel noticed that his classroom was warmer than usual. "
            "He opened two windows to let fresh air enter. "
            "After a few minutes, the room felt cooler."
        ),
        "answer": "two windows"
    },
    {
        "context": (
            "Bees transfer pollen between flowers while collecting nectar. "
            "This process helps many plants produce fruits and seeds. "
            "Without pollinators, some crops would produce less food."
        ),
        "answer": "pollen"
    },
    {
        "context": (
            "Lena saved a portion of her allowance every week. "
            "After three months, she had enough money to buy a new backpack. "
            "She also learned to plan her spending carefully."
        ),
        "answer": "After three months"
    },
    {
        "context": (
            "The river became polluted when factories released untreated waste. "
            "Fish began to disappear from the affected area. "
            "Local officials ordered the factories to improve their waste treatment."
        ),
        "answer": "untreated waste"
    },
    {
        "context": (
            "Marco studied in a quiet room every evening. "
            "He reviewed his notes and practiced answering sample questions. "
            "During the examination, he felt more confident."
        ),
        "answer": "every evening"
    },
    {
        "context": (
            "A strong typhoon damaged several houses in the coastal village. "
            "Residents moved to the evacuation center before the storm arrived. "
            "Emergency workers provided food and blankets."
        ),
        "answer": "the evacuation center"
    },
    {
        "context": (
            "Plants need sunlight to make their own food through photosynthesis. "
            "Their roots absorb water from the soil. "
            "Their leaves take in carbon dioxide from the air."
        ),
        "answer": "sunlight"
    },
    {
        "context": (
            "Nina arrived late because the bus broke down on the highway. "
            "She called her teacher to explain the delay. "
            "The teacher allowed her to complete the missed activity later."
        ),
        "answer": "the bus broke down on the highway"
    },
    {
        "context": (
            "The community placed separate bins for paper, plastic, and metal. "
            "Volunteers taught residents how to sort their waste. "
            "After several weeks, less garbage was sent to the landfill."
        ),
        "answer": "separate bins"
    },
    {
        "context": (
            "Arun compared the prices of three calculators before buying one. "
            "He chose the model that met his needs and cost less. "
            "He kept the remaining money for school supplies."
        ),
        "answer": "the model that met his needs and cost less"
    }
]

for i, case in enumerate(test_cases, start=1):
    context = case["context"]
    answer = case["answer"]

    start = context.find(answer)
    assert start != -1 and context.count(answer) == 1, (
        f"Answer span problem in test {i}: {answer}"
    )

    highlighted = (
        context[:start]
        + "<hl> " + answer + " <hl>"
        + context[start + len(answer):]
    )

    print("\n" + "=" * 55)
    print(f"TEST {i}")
    print("Expected answer:", answer)
    print("Original T5:")
    print(generate_question(base_model, base_tokenizer, highlighted))
    print("Fine-tuned T5:")
    print(generate_question(fine_model, fine_tokenizer, highlighted))


TEST 1
Expected answer: two windows
Original T5:
How many windows did Daniel open to let fresh air into his classroom?
Fine-tuned T5:
How many windows did Daniel open to let fresh air in?

TEST 2
Expected answer: pollen
Original T5:
Bees transfer what between flowers while collecting nectar?
Fine-tuned T5:
What do bees transfer between flowers while collecting nectar?

TEST 3
Expected answer: After three months
Original T5:
When did Lena have enough money to buy a new backpack?
Fine-tuned T5:
When did Lena have enough money to buy a new backpack?

TEST 4
Expected answer: untreated waste
Original T5:
What was released into the river by factories?
Fine-tuned T5:
What did factories release into the river?

TEST 5
Expected answer: every evening
Original T5:
When did Marco study in a quiet room?
Fine-tuned T5:
When did Marco study in a quiet room?

TEST 6
Expected answer: the evacuation center
Original T5:
Where did residents move to before the storm arrived?
Fine-tuned T5:
Where did resid

In [35]:
from google.colab import files
uploaded = files.upload()


Saving t5_qg_200_new_story_examples.json to t5_qg_200_new_story_examples.json


In [36]:
import json

with open(
    "t5_qg_200_new_story_examples.json",
    "r",
    encoding="utf-8"
) as f:
    story_examples = json.load(f)

print("New examples:", len(story_examples))
print("New stories:", len(set(
    item["passage_id"] for item in story_examples
)))

New examples: 200
New stories: 50


In [37]:
expanded_examples = all_examples + new_examples + story_examples

print("Total examples:", len(expanded_examples))
print(
    "Total unique passages:",
    len(set(item["passage_id"] for item in expanded_examples))
)

Total examples: 300
Total unique passages: 75


In [39]:
from collections import Counter

print("Total examples:", len(expanded_examples))
print("Unique passages:", len(set(
    ex["passage_id"] for ex in expanded_examples
)))

# Check duplicate question-answer pairs within the same passage
seen = set()
duplicates = []

for ex in expanded_examples:
    key = (
        ex["passage_id"],
        ex["question"].strip().lower(),
        ex["answer"].strip().lower()
    )
    if key in seen:
        duplicates.append(key)
    seen.add(key)

print("Duplicate question-answer pairs:", len(duplicates))

# Check that each answer appears exactly once in its passage
invalid_answers = []

for ex in expanded_examples:
    context = ex["context"]
    answer = ex["answer"]

    if context.count(answer) != 1:
        invalid_answers.append({
            "passage_id": ex["passage_id"],
            "answer": answer,
            "occurrences": context.count(answer)
        })

print("Invalid answer spans:", len(invalid_answers))

if duplicates:
    print("Duplicate examples:", duplicates[:5])

if invalid_answers:
    print("Invalid spans:", invalid_answers[:10])

Total examples: 300
Unique passages: 75
Duplicate question-answer pairs: 0
Invalid answer spans: 0


In [43]:
from collections import defaultdict

original_examples = all_examples + new_examples

original_contexts = defaultdict(set)

for ex in original_examples:
    original_contexts[ex["passage_id"]].add(ex["context"])

collisions = []

for ex in story_examples:
    pid = ex["passage_id"]

    if pid in original_contexts:
        same_context = ex["context"] in original_contexts[pid]
        collisions.append({
            "passage_id": pid,
            "same_context": same_context
        })

print("Passage ID collisions:", len(collisions))
print(collisions)

Passage ID collisions: 0
[]


In [41]:
# Give all new story examples their own unique ID prefix
for ex in story_examples:
    ex["passage_id"] = "story_" + ex["passage_id"]

# Rebuild the combined dataset
expanded_examples = all_examples + new_examples + story_examples

print("Total examples:", len(expanded_examples))
print(
    "Unique passages:",
    len(set(ex["passage_id"] for ex in expanded_examples))
)

Total examples: 300
Unique passages: 76


In [44]:
from collections import Counter

# Check answer spans
invalid_answers = [
    ex for ex in expanded_examples
    if ex["context"].count(ex["answer"]) != 1
]

# Check duplicate question-answer pairs
seen = set()
duplicates = []

for ex in expanded_examples:
    key = (
        ex["passage_id"],
        ex["question"].strip().lower(),
        ex["answer"].strip().lower()
    )

    if key in seen:
        duplicates.append(key)

    seen.add(key)

print("Invalid answer spans:", len(invalid_answers))
print("Duplicate question-answer pairs:", len(duplicates))

Invalid answer spans: 0
Duplicate question-answer pairs: 0


In [45]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Use passage IDs as groups to prevent data leakage
indices = np.arange(len(expanded_examples))
groups = np.array([
    ex["passage_id"] for ex in expanded_examples
])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(indices, groups=groups)
)

train_examples = [
    expanded_examples[i] for i in train_idx
]

val_examples = [
    expanded_examples[i] for i in val_idx
]

train_passages = {
    ex["passage_id"] for ex in train_examples
}

val_passages = {
    ex["passage_id"] for ex in val_examples
}

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Training passages:", len(train_passages))
print("Validation passages:", len(val_passages))
print("Passages overlap:", train_passages & val_passages)

Training examples: 238
Validation examples: 62
Training passages: 60
Validation passages: 16
Passages overlap: set()


In [46]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

MODEL_NAME = "valhalla/t5-base-qg-hl"
SAVE_DIR = "/content/t5-qg-finetuned-300"

# Mark the answer span for T5
def prepare_t5_example(ex):
    context = ex["context"]
    answer = ex["answer"]

    marked_context = context.replace(
        answer,
        f"<hl> {answer} <hl>",
        1
    )

    return {
        "input_text": f"generate question: {marked_context}",
        "target_text": ex["question"]
    }

train_data = Dataset.from_list([
    prepare_t5_example(ex) for ex in train_examples
])

val_data = Dataset.from_list([
    prepare_t5_example(ex) for ex in val_examples
])

print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))
print("\nSample input:")
print(train_data[0]["input_text"])
print("\nExpected question:")
print(train_data[0]["target_text"])

Training rows: 238
Validation rows: 62

Sample input:
generate question: Every evening at dusk, Mara climbed the <hl> ninety-one steps <hl> of the lighthouse to light the lamp, just as her grandfather had done before her. The lamp itself was old, but it never failed. What worried her was the harbor below, where fewer boats returned each season.

Tonight, a single fishing boat wobbled toward the rocks. Mara did not wave or shout. She turned the lamp's shutter slowly, three long flashes and one short, the way her grandfather had taught her when she was small. The boat straightened and slid into the channel between the stones.

At the dock, the fisherman looked up at the tower and lifted his cap. He had never met Mara, yet he knew exactly who was up there. The old signal had told him that the person keeping the light still remembered the old ways, and that the channel was safe to trust.

Expected question:
How many steps did Mara climb to reach the lighthouse lamp?


In [47]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("CUDA available:", torch.cuda.is_available())
print("Model loaded:", MODEL_NAME)

CUDA available: True
Model loaded: valhalla/t5-base-qg-hl


In [48]:
def tokenize_examples(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["target_text"],
        max_length=64,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_train = train_data.map(
    tokenize_examples,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    tokenize_examples,
    batched=True,
    remove_columns=val_data.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Tokenization complete.")

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Tokenization complete.


In [49]:
training_args = Seq2SeqTrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    predict_with_generate=True,
    generation_max_length=64,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_4975/1927391310.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,1.024900,0.642871
2,0.778900,0.615094
3,0.776800,0.608545
4,0.694400,0.603533
5,0.663300,0.602629


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=300, training_loss=0.7391937987009685, metrics={'train_runtime': 502.9605, 'train_samples_per_second': 2.366, 'train_steps_per_second': 0.596, 'total_flos': 166416773529600.0, 'train_loss': 0.7391937987009685, 'epoch': 5.0})

In [50]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved to:", SAVE_DIR)

Model saved to: /content/t5-qg-finetuned-300


# checkpoint 2

In [51]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_NAME = "valhalla/t5-base-qg-hl"
SAVE_DIR = "/content/t5-qg-finetuned-300"

# Load the original model for comparison
base_tokenizer = AutoTokenizer.from_pretrained(BASE_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_NAME
).to("cuda")

# Reload the newly trained model from disk
fine_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
fine_model = AutoModelForSeq2SeqLM.from_pretrained(
    SAVE_DIR
).to("cuda")

base_model.eval()
fine_model.eval()

print("Original model loaded.")
print("Fine-tuned model reloaded successfully.")

Original model loaded.
Fine-tuned model reloaded successfully.


In [52]:
def generate_question(model, tokenizer, ex):
    prepared = prepare_t5_example(ex)

    inputs = tokenizer(
        prepared["input_text"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


# Use 10 examples from held-out validation passages
for i, ex in enumerate(val_examples[:10], start=1):
    original_question = generate_question(
        base_model, base_tokenizer, ex
    )
    tuned_question = generate_question(
        fine_model, fine_tokenizer, ex
    )

    print(f"\n--- Example {i} ---")
    print("Expected:   ", ex["question"])
    print("Original:   ", original_question)
    print("Fine-tuned: ", tuned_question)


--- Example 1 ---
Expected:    What did the plants receive near the fence?
Original:    Why did Daniel help move the plants near the fence?
Fine-tuned:  Why did Daniel help move the plants near the fence?

--- Example 2 ---
Expected:    When did Daniel visit the community garden?
Original:    When does Daniel visit the community garden?
Fine-tuned:  When did Daniel visit the community garden?

--- Example 3 ---
Expected:    What tiny animals build coral reefs?
Original:    What are coral reefs made of?
Fine-tuned:  What tiny animals build coral reefs?

--- Example 4 ---
Expected:    What forms the structures of coral reefs over time?
Original:    What are coral polyps' structures made of?
Fine-tuned:  What form structures that provide shelter for many sea creatures?

--- Example 5 ---
Expected:    What do coral reef structures provide?
Original:    What do coral polyps provide?
Fine-tuned:  What do coral polyp structures provide?

--- Example 6 ---
Expected:    What kind of ocean cond

In [53]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

SOURCE_DIR = "/content/t5-qg-finetuned-300"
BACKUP_DIR = "/content/drive/MyDrive/Thesis_T5/t5-qg-finetuned-300"

os.makedirs(os.path.dirname(BACKUP_DIR), exist_ok=True)

shutil.copytree(
    SOURCE_DIR,
    BACKUP_DIR,
    dirs_exist_ok=True
)

print("Model backup saved to:", BACKUP_DIR)

Mounted at /content/drive
Model backup saved to: /content/drive/MyDrive/Thesis_T5/t5-qg-finetuned-300


In [54]:
import json
import os

PROJECT_DIR = "/content/drive/MyDrive/Thesis_T5"
os.makedirs(PROJECT_DIR, exist_ok=True)

with open(
    os.path.join(PROJECT_DIR, "t5_qg_300_dataset.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        expanded_examples,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Dataset backup saved.")
print("Examples:", len(expanded_examples))

Dataset backup saved.
Examples: 300
